In [ ]:
import requests
import json
import time
import os
import re
import csv
import concurrent.futures
import numpy as np
import random
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import zipfile

# ==========================================
#        CONFIGURATION (GLOBAL SETTINGS)
# ==========================================

IDS_FILE = "riyadh_parking_ids.txt"
JSONL_FILE = "riyadh_parking_data.jsonl"
CSV_FILENAME = "riyadh_parking_lots.csv"
ZIP_NAME = "riyadh_parking_complete.zip"

# FULL RIYADH BOUNDARIES
LAT_MIN, LAT_MAX = 24.3500, 25.2500
LON_MIN, LON_MAX = 46.2500, 47.2000

# GRID SETTINGS (Tuned for Parking)
SCOUT_STEP = 0.01      # 1.1km
SCOUT_RADIUS = 1200
DRILL_STEP = 0.002     # ~220m
DRILL_RADIUS = 300

BATCH_SIZE = 50
MAX_WORKERS = 20

API_KEY = "REDACTED_API_KEY"
HEADERS = {"User-Agent": "2GIS/5.0 (Android; Mobile)", "Accept": "application/json"}

# OPTIMIZED SESSION
session = requests.Session()
session.headers.update(HEADERS)
adapter = HTTPAdapter(
    pool_connections=MAX_WORKERS*2,
    pool_maxsize=MAX_WORKERS*2,
    max_retries=Retry(total=3, backoff_factor=0.5)
)
session.mount("https://", adapter)

# --- PHASE 1 LOGIC ---

def fetch_parking_ids(lat, lon, radius):
    found = set()
    page = 1
    while True:
        url = "https://catalog.api.2gis.ru/3.0/items"
        params = {
            "key": API_KEY,
            "point": f"{lon},{lat}",
            "radius": radius,
            "type": "parking",
            "fields": "items.id",
            "page_size": 50,
            "page": page
        }
        try:
            resp = session.get(url, params=params, timeout=5)
            if resp.status_code != 200: break

            data = resp.json().get('result', {})
            items = data.get('items', [])
            total = data.get('total', 0)

            if not items: break

            for item in items:
                raw_id = str(item['id'])
                found.add(raw_id.split('_')[0])

            if len(items) < 50: break
            page += 1
        except: break

    return found, total

def worker_scan(sector):
    lat, lon, level = sector
    # SCOUT
    if level == 1:
        try:
            ids, total = fetch_parking_ids(lat, lon, SCOUT_RADIUS)
            if total == 0: return None
            # If found, generate drill tasks
            micro_lats = np.arange(lat, lat + SCOUT_STEP, DRILL_STEP)
            micro_lons = np.arange(lon, lon + SCOUT_STEP, DRILL_STEP)
            return [(mlat, mlon, 2) for mlat in micro_lats for mlon in micro_lons]
        except: return None
    # DRILL
    if level == 2:
        ids, _ = fetch_parking_ids(lat, lon, DRILL_RADIUS)
        return ids

def run_phase_1():
    print("\n--- PHASE 1: SCANNING FOR PARKING LOTS ---")

    scout_lats = np.arange(LAT_MIN, LAT_MAX, SCOUT_STEP)
    scout_lons = np.arange(LON_MIN, LON_MAX, SCOUT_STEP)
    tasks = [(lat, lon, 1) for lat in scout_lats for lon in scout_lons]
    random.shuffle(tasks)

    print(f"Sectors to Scout: {len(tasks)}")

    existing_ids = set()
    if os.path.exists(IDS_FILE):
        print("Resuming from existing ID list...")
        with open(IDS_FILE, "r") as f:
            for line in f: existing_ids.add(line.strip())

    with open(IDS_FILE, "a") as f:
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_task = {executor.submit(worker_scan, t): t for t in tasks}
            pbar = tqdm(total=len(tasks), unit="sector", desc="Scanning")

            while future_to_task:
                done, _ = concurrent.futures.wait(future_to_task, return_when=concurrent.futures.FIRST_COMPLETED)
                for future in done:
                    task_data = future_to_task.pop(future)
                    try:
                        result = future.result()
                        if task_data[2] == 1: # Scout
                            pbar.update(1)
                            if isinstance(result, list):
                                for dt in result:
                                    future_to_task[executor.submit(worker_scan, dt)] = dt
                        elif task_data[2] == 2: # Drill
                            if result:
                                new = result - existing_ids
                                if new:
                                    for uid in new:
                                        f.write(f"{uid}\n")
                                        existing_ids.add(uid)
                                    f.flush()
                                    pbar.set_description(f"Drilling (Lots: {len(existing_ids)})")
                    except: pass
            pbar.close()
    print(f"Found {len(existing_ids)} unique Parking Lots.")

if __name__ == "__main__":
    run_phase_1()

In [ ]:
import requests
import json
import time
import os
import concurrent.futures
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- CONFIGURATION ---
IDS_FILE = "riyadh_parking_ids.txt"
JSONL_FILE = "riyadh_parking_data_v2.jsonl" # New file for rich data
BATCH_SIZE = 50
MAX_WORKERS = 20
API_KEY = "REDACTED_API_KEY"

# --- ALL FIELDS ARE ADDED HERE ---
FIELDS = (
    "items.geometry.centroid,items.geometry.selection,items.address,items.name_ex,"
    "items.adm_div,items.capacity,items.is_paid,items.access,items.subtype,"
    "items.schedule,items.purpose_name,items.point,items.context,"
    "items.for_trucks,items.is_incentive,items.purpose"
)

HEADERS = {"User-Agent": "2GIS/5.0", "Accept": "application/json"}
session = requests.Session()
session.headers.update(HEADERS)
adapter = HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS, max_retries=Retry(total=3))
session.mount("https://", adapter)

def id_gen():
    batch = []
    if not os.path.exists(IDS_FILE):
        print("Error: IDs file missing. Run Phase 1 first.")
        return
    with open(IDS_FILE, "r") as f:
        for line in f:
            clean = line.strip().split('_')[0]
            if clean:
                batch.append(clean)
                if len(batch) >= BATCH_SIZE:
                    yield batch
                    batch = []
    if batch: yield batch

def fetch_details(batch):
    try:
        url = "https://catalog.api.2gis.ru/3.0/items/byid"
        resp = session.get(url, params={"key": API_KEY, "id": ",".join(batch), "fields": FIELDS, "locale": "en_SA"}, timeout=10)
        return resp.json().get('result', {}).get('items', []) if resp.status_code == 200 else []
    except: return []

def run_phase_2_update():
    print("\n--- PHASE 2: RE-DOWNLOADING WITH TRUCK/INCENTIVE DATA ---")

    # We write to a NEW file to avoid mixing old (incomplete) data with new data
    if os.path.exists(JSONL_FILE):
        os.remove(JSONL_FILE)

    total = sum(1 for _ in open(IDS_FILE))
    print(f"Fetching rich details for {total} parking lots...")

    with open(JSONL_FILE, "a", encoding="utf-8") as f_out:
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(fetch_details, batch): batch for batch in id_gen()}

            for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), unit="batch"):
                items = future.result()
                if items:
                    for item in items:
                        f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
                    f_out.flush()

if __name__ == "__main__":
    run_phase_2_update()

In [ ]:
import json
import csv
import re
import os
from tqdm import tqdm
import zipfile

# --- CONFIGURATION ---
JSONL_FILE = "riyadh_parking_data_v2.jsonl" # Reading the NEW file
CSV_FILENAME = "riyadh_parking_lots_detailed.csv"
ZIP_NAME = "riyadh_parking_detailed.zip"

def extract_geo(wkt):
    if not wkt: return None, None
    m = re.search(r"POINT\(([\d\.]+) ([\d\.]+)\)", wkt)
    return (m.group(2), m.group(1)) if m else (None, None)

def parse_full_schedule(schedule_obj):
    """
    Extracts every single working hour detail.
    """
    if not schedule_obj:
        return "Unknown"

    # 1. Check 24/7
    if schedule_obj.get("is_24x7"):
        return "24 Hours Daily"

    # 2. Iterate Days
    days_output = []
    # 2GIS keys for days
    day_map = {
        "Mon": "Mon", "Tue": "Tue", "Wed": "Wed", "Thu": "Thu",
        "Fri": "Fri", "Sat": "Sat", "Sun": "Sun"
    }

    for key, label in day_map.items():
        day_info = schedule_obj.get(key)
        if day_info and "working_hours" in day_info:
            hours = day_info["working_hours"]
            # Join multiple slots if they exist (e.g., 08-12 and 14-18)
            slots = [f"{h.get('from')}-{h.get('to')}" for h in hours]
            if slots:
                days_output.append(f"{label}: {', '.join(slots)}")

    if not days_output:
        return "See Comments" # Sometimes hours are in comments, rare

    return " | ".join(days_output)

def run_phase_3_detailed():
    print("\n--- PHASE 3: CONVERTING WITH FULL SCHEDULES ---")

    headers = [
        "ID", "District", "Address", "Latitude", "Longitude",
        "Type", "Subtype", "Capacity", "Is_Paid", "Access",
        "Purpose", "For_Trucks", "Is_Incentive",
        "Full_Schedule",  # <--- The Detailed Schedule Column
        "Polygon_WKT"
    ]

    f_out = open(CSV_FILENAME, "w", encoding="utf-8", newline="")
    writer = csv.writer(f_out)
    writer.writerow(headers)

    count = 0
    if os.path.exists(JSONL_FILE):
        with open(JSONL_FILE, "r", encoding="utf-8") as f_in:
            for line in tqdm(f_in, desc="Converting"):
                try:
                    d = json.loads(line)

                    # Geometry
                    geo = d.get("geometry", {})
                    wkt = geo.get("selection", "")
                    centroid = geo.get("centroid", "")
                    lat, lon = extract_geo(centroid)

                    # District
                    district = ""
                    for div in d.get("adm_div", []):
                        if div.get("type") == "district":
                            district = div.get("name", "")
                            break

                    # Schedule
                    sched_text = parse_full_schedule(d.get("schedule", {}))

                    row = [
                        d.get("id"),
                        district,
                        d.get("address_name", "") or d.get("full_name", ""),
                        lat, lon,
                        d.get("type", "parking"),
                        d.get("subtype", ""),
                        d.get("capacity", {}).get("total", ""),
                        d.get("is_paid", ""),
                        d.get("access", ""),

                        # New Fields (Now they will have data!)
                        d.get("purpose", ""),
                        d.get("for_trucks", ""),
                        d.get("is_incentive", ""),
                        sched_text,

                        wkt
                    ]
                    writer.writerow(row)
                    count += 1
                except: continue

    f_out.close()

    with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(CSV_FILENAME)

    print(f"\n✅ SUCCESS! Extracted {count} lots.")
    print(f"Download: {ZIP_NAME}")

if __name__ == "__main__":
    run_phase_3_detailed()

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv("riyadh_parking_lots_detailed.csv")
df.sample(1)

In [ ]:
df["Full_Schedule"].value_counts()

In [ ]:
df[df["Capacity"]==641]

In [ ]:
import requests

API_KEY = "REDACTED_API_KEY"

# Center of Al Olaya District
LAT = 24.6958
LON = 46.6853
RADIUS = 1500  # 1.5km radius (Small enough for the API to answer)

def check_local_count():
    url = "https://catalog.api.2gis.ru/3.0/items"
    params = {
        "key": API_KEY,
        "point": f"{LON},{LAT}",
        "radius": RADIUS,
        "type": "building",
        "fields": "items.id",
        "page_size": 1
    }

    try:
        response = requests.get(url, params=params)
        data = response.json()
        total = data.get('result', {}).get('total', 0)

        print(f"--- OFFICIAL API LOCAL CHECK ---")
        print(f"Target: Al Olaya (1.5km Radius)")
        print(f"Official Count: {total}")
        return total
    except Exception as e:
        print(e)
        return 0

if __name__ == "__main__":
    api_count = check_local_count()

In [ ]:
import pandas as pd
import glob
import math

# Center of Al Olaya (Same as API check)
CENTER_LAT = 24.6958
CENTER_LON = 46.6853
RADIUS_KM = 1.5

def haversine_distance(lat1, lon1, lat2, lon2):
    # Earth radius in kilometers
    R = 6371.0

    # Convert coords to radians
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi / 2)**2 + \
        math.cos(phi1) * math.cos(phi2) * \
        math.sin(dlambda / 2)**2

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

def check_csv_count():
    files = glob.glob("riyadh_buildings_part*.csv")

    if not files:
        print("Error: No CSV files found.")
        return

    count = 0
    total_scanned = 0

    print("Scanning your local CSVs...")

    for f in files:
        try:
            # Load data
            df = pd.read_csv(f, usecols=["Latitude", "Longitude"])

            # 1. Fast Filter (Bounding Box) to speed up processing
            # 1.5km is approx 0.015 degrees. We use 0.02 buffer.
            df = df[
                (df.Latitude > CENTER_LAT - 0.02) & (df.Latitude < CENTER_LAT + 0.02) &
                (df.Longitude > CENTER_LON - 0.02) & (df.Longitude < CENTER_LON + 0.02)
            ]

            # 2. Precise Filter (Haversine Distance)
            for _, row in df.iterrows():
                dist = haversine_distance(CENTER_LAT, CENTER_LON, row['Latitude'], row['Longitude'])
                if dist <= RADIUS_KM:
                    count += 1

        except Exception as e:
            print(f"Skipping file due to error: {e}")

    print("\n" + "="*30)
    print(f"OFFICIAL API COUNT:  5,924")
    print(f"YOUR SCRAPED COUNT:  {count:,}")
    print("="*30)

    # Verdict Logic
    diff = abs(count - 5924)
    percentage = (count / 5924) * 100

    if diff < 100:
        print("✅ PERFECT MATCH (99-100% Accuracy)")
    elif diff < 500:
        print(f"✅ EXCELLENT ({percentage:.1f}% Accuracy). Minor timing differences are normal.")
    elif count > 5924:
        print(f"✅ SUPERIOR DATA. You captured {count - 5924} buildings that the API hides at zoom level.")
    else:
        print(f"⚠️ MISSING DATA. You are missing {5924 - count} buildings in this district.")

if __name__ == "__main__":
    check_csv_count()

In [ ]:
import pandas as pd
import glob

def check_duplicates():
    print("Checking for duplicates in final dataset...")
    files = glob.glob("riyadh_buildings_part*.csv")

    # Load just IDs
    df_list = []
    for f in files:
        try:
            df = pd.read_csv(f, usecols=["ID"])
            df_list.append(df)
        except: pass

    full_df = pd.concat(df_list)

    total_rows = len(full_df)
    unique_ids = full_df["ID"].nunique()
    duplicates = total_rows - unique_ids

    print(f"Total Rows:   {total_rows:,}")
    print(f"Unique IDs:   {unique_ids:,}")
    print(f"Duplicates:   {duplicates:,}")

    if duplicates == 0:
        print("\n✅ DATASET IS CLEAN. Your 10k count is real.")
    else:
        print(f"\n⚠️ FOUND DUPLICATES. Real count is {unique_ids:,}.")

if __name__ == "__main__":
    check_duplicates()